# Caso A — Abastecimiento · EDA sobre la tabla maestra

En este notebook exploramos la tabla maestra del Caso A antes de modelar. Para ello cruzamos las ventas con el catálogo de productos, el maestro de tiendas, el inventario y la tendencia de referencia, y agregamos el resultado a grano semanal por SKU-tienda. Sobre esa tabla única analizamos el tipado de cada variable, sus estadísticos, la multicolinealidad (VIF), las correlaciones y qué predictores discriminan la demanda (tests e información mutua).

Las fuentes se cargan por el catálogo de Kedro (`data/01_raw`), sin `pd.read_csv` sueltos, y todo el análisis reutiliza `tostao_ml`: el notebook orquesta y narra, no reimplementa lógica. Para reproducirlo basta con reiniciar el kernel y ejecutar de arriba abajo (`Restart & Run All`), ya que el proceso es determinista.

## 1. Construcción de la tabla maestra (cruce de fuentes)

`ventas ⨝ catálogo(producto) ⨝ maestro_tiendas(tienda) ⨝ inventario(tienda+producto) ⨝ ground_truth(tienda+producto)`, luego agrego a grano **semanal**.

In [ ]:
from pathlib import Path
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project
from tostao_ml.cases import masters
from tostao_ml.framework.profiling import profile_dataset, build_eda_figures

PROJECT = Path.cwd().parents[1] if Path.cwd().name.startswith('caso') else Path.cwd()
bootstrap_project(PROJECT)
with KedroSession.create(project_path=PROJECT) as session:
    catalog = session.load_context().catalog
    ventas = catalog.load('a_ventas_historicas'); cat = catalog.load('a_catalogo_productos')
    tiendas = catalog.load('a_maestro_tiendas'); inv = catalog.load('a_inventario_actual')
    trends = catalog.load('a_ground_truth_trends')
master_d, join_report = masters.build_master_a(ventas, cat, tiendas, inv, trends)
master = masters.aggregate_weekly_a(master_d)  # grano de forecast: semana x SKU x tienda


In [ ]:
print('Master:', master.shape)
print('Cobertura de cruces:', join_report)
master.head()

La **cobertura de cruces** confirma la integridad referencial: la proporción de filas del hecho que encontró match en cada fuente unida.

## 2. Perfilado estadístico (motor de EDA reutilizable)

In [ ]:
profile = profile_dataset(master, name='Caso A — Abastecimiento', target='unidades_vendidas')
print('Tipos:')
for c, k in profile.types.items():
    print(f'  {c:28s} {k.value}')
profile.univariate.round(3)

### Mini-conclusiones autogeneradas (cifras reales del run)

In [ ]:
from IPython.display import Markdown
Markdown(profile.narrative.to_markdown())

## 3. Multicolinealidad y correlaciones

In [ ]:
figs = build_eda_figures(master, profile)
display(profile.vif.round(3).to_frame('VIF'))
figs.get('correlation_heatmap')

## 4. Relación con el target e información mutua

In [ ]:
display(profile.mutual_information.round(4).to_frame('MI'))
profile.bivariate.round(4)

In [ ]:
for name, fig in figs.items():
    if name.startswith(('dist__', 'target__')):
        fig.show()

## 5. Conclusión

El EDA sobre la **tabla maestra cruzada** (no fuente por fuente) revela el tipado de cada variable, la multicolinealidad (VIF), las correlaciones y qué features discriminan el target (tests + información mutua). Estas conclusiones guían el feature engineering y la elección de modelo del caso.